In [1]:
import os 
import json
from dotenv import load_dotenv
from openai  import OpenAI
import gradio as gr

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')

groq_url = "https://api.groq.com/openai/v1"

groq = OpenAI(api_key=groq_api_key,base_url=groq_url)
MODEL = "llama-3.1-8b-instant"

In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
#gradio call back fn 
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

# gr.ChatInterface(fn=chat, type="messages").launch()

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

In [5]:
ticket_prices = {"delhi":"$440","mumbai":"$500","bangalore":"$600","chennai":"$300"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [6]:
get_ticket_price("bangalore")

Tool called for city bangalore


'The price of a ticket to bangalore is $600'

In [7]:
#need to tell llm,it can use tools,the way to tell it is by json,since model trained on lots of jsons

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [8]:
# included in a list of tools:

tools = [{"type": "function", "function": price_function}]

## LLM to use our tool 
- two parts:
1. when llm is called , need to pass this json, llm needs to know about this tool
2. when llm responds ,need to detect if its looking to run the tool,if so,need to call the tool ,get the answer and send it back as new message with whole convo history to llm as second call

In [9]:
#new chat callback
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message) #message asking for tool to be called
        messages.append(response) #response from the tool call
        # for message in messages:
        #     print(messages)
        response = groq.chat.completions.create(model=MODEL,messages=messages)
        
    return response.choices[0].message.content

In [10]:
##still llm cant make multiple tool calls => message.tool_calls[0]

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response


In [11]:
# view = gr.ChatInterface(fn=chat)
# view.launch()

In [12]:
# view.close()

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [13]:
##it doesnt support llm making more than one set of tool call sequentially
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [14]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [15]:
# view = gr.ChatInterface(fn=chat)
# view.launch()

In [16]:
# view.close()

In [35]:
##to support multiple tool call requests
# def chat(message, history):
#     history = [{"role":h["role"], "content":h["content"]} for h in history]
#     messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
#     response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

#     while response.choices[0].finish_reason=="tool_calls":
#         message = response.choices[0].message
#         responses = handle_tool_calls(message)
#         messages.append(message)
#         messages.extend(responses)
#         response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
#     return response.choices[0].message.content

def chat(message, history):

    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    acknowledgements = [
        "ok", "okay", "thanks", "thank you",
        "cool", "great", "nice"
    ]

    # Disable tools for acknowledgement messages
    if message.lower().strip() in acknowledgements:

        response = groq.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        return response.choices[0].message.content

    # Normal tool flow
    response = groq.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":

        message = response.choices[0].message

        responses = handle_tool_calls(message)

        messages.append(message)
        messages.extend(responses)

        response = groq.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content


In [18]:
# view = gr.ChatInterface(fn=chat)
# view.launch()

In [19]:
# view.close()

## TOOL CALLING WITH SQL INTEGRATION

In [20]:
import sqlite3

In [21]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    sql = conn.cursor()
    sql.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [22]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        sql = conn.cursor()
        sql.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = sql.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [23]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $699.0'

In [29]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        sql = conn.cursor()
        sql.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [30]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [31]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [ ]:
view = gr.ChatInterface(fn=chat)
view.launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Tokyo
DATABASE TOOL CALLED: Getting price for Sydney
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris


In [37]:
view.close()

Closing server running on port: 7881


## To go multi-modal with free, open-source models, you’ll need alternatives for DALL-E 3 (Image Generation) and OpenAI TTS (Text-to-Speech). Since you are already using Ollama for text, you can complement it with specialized local libraries or Hugging Face models.

1. Image Generation (DALL-E 3 Alternative)
The current gold standard for open-source image generation is Stable Diffusion XL (SDXL) or the newer FLUX.1-schnell.

To run these locally in Python, use the diffusers library.

Installation: pip install diffusers transformers accelerate

In [38]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
from huggingface_hub import login

# Option A: Manual login (it will ask for your token)
# login() 

# Option B: Automatic (if you have the token in your .env or a variable)
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

# 1. Load without float16 for CPU compatibility
pipe = StableDiffusionXLPipeline.from_pretrained(
    "segmind/SSD-1B", 
    use_safetensors=True
)

pipe.to("cpu")

def artist(city):
    prompt = f"A vibrant pop-art vacation poster of {city}, featuring iconic landmarks and tourist spots, highly detailed, 8k"
    
    # SSD-1B works best between 20-30 steps
    # guidance_scale 7.0 to 9.0 is usually the sweet spot for this model
    image = pipe(
        prompt=prompt, 
        num_inference_steps=12, # Fewer steps = much faster
        guidance_scale=7.0,
        width=512,              # Smaller width
        height=512              # Smaller height
    ).images[0]
    
    return image

image = artist("New York City")
display(image)

[transformers] `CLIPImageProcessor` requires torchvision (not installed); falling back to `CLIPImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `CLIPImageProcessorPil` directly to silence this warning.
[transformers] `SiglipImageProcessor` requires torchvision (not installed); falling back to `SiglipImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `SiglipImageProcessorPil` directly to silence this warning.
Fetching 18 files:  17%|█▋        | 3/18 [00:00<00:01,  9.64it/s]c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--segmind--SSD-1B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be di

## 2. Text-to-Speech (GPT-4o-mini-tts Alternative)
For speech, Kokoro-82M is currently the best "small but mighty" open-source model. It is incredibly fast and sounds very natural. Another classic choice is pyttsx3 (completely offline, uses system voices) or coqui-tts.

Option A: Kokoro (High Quality)
Installation: pip install kokoro-onnx soundfile

In [1]:
from kokoro_onnx import Kokoro
import soundfile as sf

# Note: You'll need the kokoro.onnx file and voices.bin from Hugging Face
kokoro = Kokoro("kokoro-v0_19.onnx", "voices.bin")

def talker(message):
    samples, sample_rate = kokoro.create(message, voice="af_bella", speed=1.0)
    # To save or play:
    sf.write("output.wav", samples, sample_rate)
    return "output.wav"

FileNotFoundError: Voices file not found at voices.bin
You can download the voices file using the following command:
wget https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin

In [4]:
import pyttsx3

def talker(message):
    engine = pyttsx3.init()
    engine.say(message)
    engine.runAndWait()

In [6]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    # image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    # if cities:
    #     image = artist(cities[0])
    
    # return history, voice, image
    return history, voice

In [7]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))